In [1]:
import json
import math
import re
import sys
from pathlib import Path

import numpy as np
import torch
import torchaudio
from torch.utils.data import DataLoader

REPO_ROOT = Path("/home/amin/Research/Representation/Movie/cav-mae-sync")
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import dataloader_sync
from dataloader_sync import train_collate_fn
import models

DEVICE = "cpu"
print("Device:", DEVICE)

/home/amin/miniconda3/envs/cav-mae-sync/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [2]:
DEFAULT_TOTAL_FRAMES = 16
DEFAULT_AUDIO_MEAN = -5.081
DEFAULT_AUDIO_STD = 4.4849
DEFAULT_MODEL_PATH = REPO_ROOT / "pretrained_models" / "cav_mae_sync.pth"
DEFAULT_OUTPUTS_ROOT = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings/cav-mae-sync")


In [3]:
# Configure inputs
VIDEO_PATH = Path("/home/amin/Research/Representation/Movie/data/7T_MOVIE1_CC1_v2.mp4")
VIDEO_ID = VIDEO_PATH.stem
WAV_PATH = VIDEO_PATH.with_suffix(".wav")

BASE_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh/Data")

FORCE_CPU = True
LABELS = ["m0"]

SEGMENT_CONFIGS = {
    "2s": {
        "segment_seconds": 2,
        "target_length": 192,
        "use_presegmented": True,
        "prefix": "segment_2s",
    },
    "4s": {
        "segment_seconds": 4,
        "target_length": 416,
        "use_presegmented": False,
        "prefix": "segment_4s",
    },
}

# Change this list to run only one duration (e.g. ["2s"]).
RUN_CONFIGS = ["2s"]


In [5]:
import shutil
import subprocess


def extract_wav(video_path: Path, wav_path: Path):
    cmd = [
        "ffmpeg",
        "-i",
        str(video_path),
        "-vn",
        "-ac",
        "1",
        "-ar",
        "16000",
        "-y",
        str(wav_path),
    ]
    print("[ffmpeg] extracting wav:", " ".join(cmd))
    subprocess.run(cmd, check=True, stderr=subprocess.DEVNULL)


def extract_wav_segment(video_path: Path, start_s: float, duration_s: float, wav_path: Path):
    cmd = [
        "ffmpeg",
        "-ss",
        f"{start_s}",
        "-t",
        f"{duration_s}",
        "-i",
        str(video_path),
        "-vn",
        "-ac",
        "1",
        "-ar",
        "16000",
        "-y",
        str(wav_path),
    ]
    print("[ffmpeg] extracting wav segment:", " ".join(cmd))
    subprocess.run(cmd, check=True, stderr=subprocess.DEVNULL)


def extract_frames_for_video(
    *,
    video_path: Path,
    wav_path: Path,
    frames_root: Path,
    video_id: str,
    total_frames: int,
):    
    frames_dir = frames_root / video_id
    if frames_dir.exists():
        return frames_dir

    frames_dir.mkdir(parents=True, exist_ok=True)
    tmp_dir = frames_dir / "_tmp_frames"
    tmp_dir.mkdir(parents=True, exist_ok=True)

    waveform, sr = torchaudio.load(wav_path)
    duration_s = float(waveform.shape[1]) / float(sr)
    fps = total_frames / max(duration_s, 1e-6)

    cmd = [
        "ffmpeg",
        "-i",
        str(video_path),
        "-vf",
        f"fps={fps}",
        "-vframes",
        str(total_frames),
        "-q:v",
        "2",
        str(tmp_dir / "frame_%03d.jpg"),
    ]
    print("[ffmpeg] extracting frames:", " ".join(cmd))
    subprocess.run(cmd, check=True, stderr=subprocess.DEVNULL)

    frame_files = sorted(tmp_dir.glob("frame_*.jpg"))
    if len(frame_files) == 0:
        raise RuntimeError(f"No frames extracted for {video_id}")
    if len(frame_files) < total_frames:
        print(f"Warning: expected {total_frames} frames, got {len(frame_files)}; padding with last frame")
        last_frame = frame_files[-1]
        for idx in range(len(frame_files) + 1, total_frames + 1):
            dst = tmp_dir / f"frame_{idx:03d}.jpg"
            shutil.copyfile(last_frame, dst)
        frame_files = sorted(tmp_dir.glob("frame_*.jpg"))

    for idx, src in enumerate(frame_files[:total_frames]):
        frame_dir = frames_dir / f"frame_{idx}"
        frame_dir.mkdir(parents=True, exist_ok=True)
        dst = frame_dir / f"{video_id}.jpg"
        src.replace(dst)

    for extra in tmp_dir.glob("*"):
        extra.unlink()
    tmp_dir.rmdir()
    return frames_dir


def extract_frames_for_clip(
    *,
    video_path: Path,
    start_s: float,
    duration_s: float,
    frames_root: Path,
    video_id: str,
    total_frames: int,
):    
    frames_dir = frames_root / video_id
    if frames_dir.exists():
        return frames_dir

    frames_dir.mkdir(parents=True, exist_ok=True)
    tmp_dir = frames_dir / "_tmp_frames"
    tmp_dir.mkdir(parents=True, exist_ok=True)

    fps = total_frames / max(duration_s, 1e-6)

    cmd = [
        "ffmpeg",
        "-ss",
        f"{start_s}",
        "-t",
        f"{duration_s}",
        "-i",
        str(video_path),
        "-vf",
        f"fps={fps}",
        "-vframes",
        str(total_frames),
        "-q:v",
        "2",
        str(tmp_dir / "frame_%03d.jpg"),
    ]
    print("[ffmpeg] extracting frames (clip):", " ".join(cmd))
    subprocess.run(cmd, check=True, stderr=subprocess.DEVNULL)

    frame_files = sorted(tmp_dir.glob("frame_*.jpg"))
    if len(frame_files) == 0:
        raise RuntimeError(f"No frames extracted for {video_id}")
    if len(frame_files) < total_frames:
        print(f"Warning: expected {total_frames} frames, got {len(frame_files)}; padding with last frame")
        last_frame = frame_files[-1]
        for idx in range(len(frame_files) + 1, total_frames + 1):
            dst = tmp_dir / f"frame_{idx:03d}.jpg"
            shutil.copyfile(last_frame, dst)
        frame_files = sorted(tmp_dir.glob("frame_*.jpg"))

    for idx, src_frame in enumerate(frame_files[:total_frames]):
        frame_dir = frames_dir / f"frame_{idx}"
        frame_dir.mkdir(parents=True, exist_ok=True)
        dst = frame_dir / f"{video_id}.jpg"
        src_frame.replace(dst)

    for extra in tmp_dir.glob("*"):
        extra.unlink()
    tmp_dir.rmdir()
    return frames_dir


def extract_part_number(filename: Path):
    match = re.search(r"part[_-]?(\d+)", filename.stem, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None


def collect_segment_pairs(base_dir: Path, run_range=range(1, 19)):
    pairs = []
    for run_idx in run_range:
        vid_chunk_dir = base_dir / f"Video{run_idx}" / f"Video{run_idx}_chunks"
        aud_chunk_dir = base_dir / f"Audio{run_idx}" / f"Audio{run_idx}_chunks"

        if not vid_chunk_dir.exists() or not aud_chunk_dir.exists():
            print(f"Warning: Missing dir for run {run_idx}")
            continue

        vids = sorted(vid_chunk_dir.glob("*.mp4"), key=lambda p: extract_part_number(p) or -1)
        auds = sorted(aud_chunk_dir.glob("*.wav"), key=lambda p: extract_part_number(p) or -1)
        if len(vids) == 0 or len(auds) == 0:
            print(f"Warning: No chunks found for run {run_idx}")
            continue

        vid_parts = {extract_part_number(v): v for v in vids if extract_part_number(v) is not None}
        aud_parts = {extract_part_number(a): a for a in auds if extract_part_number(a) is not None}

        common_parts = sorted(set(vid_parts.keys()) & set(aud_parts.keys()))
        for part_num in common_parts:
            pairs.append((vid_parts[part_num], aud_parts[part_num]))

        print(f"Run {run_idx}: {len(common_parts)} paired chunks")

    print(f"Total paired 2s segments: {len(pairs)}")
    return pairs


def build_dataset_entries_2s(frames_root: Path):
    """Build 2s dataset entries, only extracting frames if not cached."""
    data_entries = []
    pairs = collect_segment_pairs(BASE_DIR)

    for idx, (video_path, wav_path) in enumerate(pairs):
        segment_id = f"{SEGMENT_CONFIGS['2s']['prefix']}_{idx:04d}"
        frames_dir = frames_root / segment_id

        # Check if frames already exist
        if frames_dir.exists() and len(list(frames_dir.glob("frame_*"))) == DEFAULT_TOTAL_FRAMES:
            pass
        else:
            extract_frames_for_video(
                video_path=video_path,
                wav_path=wav_path,
                frames_root=frames_root,
                video_id=segment_id,
                total_frames=DEFAULT_TOTAL_FRAMES,
            )

        data_entries.append({
            "wav": str(wav_path),
            "labels": ",".join(LABELS),
            "video_id": segment_id,
            "video_path": str(frames_dir),
        })

    print(f"✓ Prepared {len(data_entries)} 2s segments")
    return data_entries


def build_dataset_entries_4s(video_path, wav_path, frames_root, segments_root, segment_seconds, prefix):
    """Build 4s dataset entries, only extracting if not cached."""
    if not wav_path.exists():
        extract_wav(video_path, wav_path)

    waveform, sr = torchaudio.load(wav_path)
    duration_s = float(waveform.shape[1]) / float(sr)
    num_segments = int(math.floor(duration_s / segment_seconds))

    data_entries = []
    for idx in range(num_segments):
        start_s = idx * segment_seconds
        segment_id = f"{prefix}_{idx:04d}"
        segment_wav = segments_root / f"{segment_id}.wav"
        frames_dir = frames_root / segment_id

        # Check if frames cached
        if frames_dir.exists() and len(list(frames_dir.glob("frame_*"))) == DEFAULT_TOTAL_FRAMES:
            pass
        else:
            if not segment_wav.exists():
                extract_wav_segment(video_path, start_s, segment_seconds, segment_wav)

            extract_frames_for_clip(
                video_path=video_path,
                start_s=start_s,
                duration_s=segment_seconds,
                frames_root=frames_root,
                video_id=segment_id,
                total_frames=DEFAULT_TOTAL_FRAMES,
            )

        data_entries.append({
            "wav": str(segment_wav),
            "labels": ",".join(LABELS),
            "video_id": segment_id,
            "video_path": str(frames_dir),
        })

    print(f"✓ Prepared {num_segments} 4s segments")
    return data_entries


def resolve_device():
    if FORCE_CPU:
        return torch.device("cpu")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def infer_register_tokens(state_dict):
    for key in ("register_tokens", "module.register_tokens"):
        if key in state_dict:
            tokens = state_dict[key]
            return tokens.shape[0] // 2
    return 4


def _resize_pos_embed(state_dict, model_state_dict, key):
    if key not in state_dict or key not in model_state_dict:
        return
    src = state_dict[key]
    dst = model_state_dict[key]
    src_shape = tuple(src.shape)
    dst_shape = tuple(dst.shape)
    if src_shape == dst_shape:
        return
    if src.ndim != 3 or dst.ndim != 3:
        print(f"[warn] skip resizing {key}: unexpected shape {src_shape} -> {dst_shape}")
        return
    # Interpolate over sequence length to fit new target length.
    src = src.permute(0, 2, 1)
    src = torch.nn.functional.interpolate(src, size=dst.shape[1], mode="linear", align_corners=False)
    state_dict[key] = src.permute(0, 2, 1)
    print(f"[info] resized {key} from {src_shape} to {dst_shape}")


In [6]:
def run_config(name: str):
    """
    Extract canonical CAV-MAE-Sync embeddings per CAV-MAE-Sync paper.
    Uses forward_feat to get CLS tokens (global tokens) for audio and video.
    Saves per-frame embeddings (num_segments, 16, 768) - CANONICAL format.
    """
    cfg = SEGMENT_CONFIGS[name]
    output_root = DEFAULT_OUTPUTS_ROOT / name
    output_root.mkdir(parents=True, exist_ok=True)

    frames_root = output_root / "frames_sync"
    segments_root = output_root / "segments"
    label_csv = output_root / "labels_dummy.csv"
    data_json = output_root / f"{VIDEO_ID}_{name}_sync.json"

    label_csv.write_text("mid,index,display_name\n" + "".join(f"{mid},{idx},{mid}\n" for idx, mid in enumerate(LABELS)))

    if cfg["use_presegmented"]:
        data_entries = build_dataset_entries_2s(frames_root)
    else:
        segments_root.mkdir(parents=True, exist_ok=True)
        data_entries = build_dataset_entries_4s(
            video_path=VIDEO_PATH,
            wav_path=WAV_PATH,
            frames_root=frames_root,
            segments_root=segments_root,
            segment_seconds=cfg["segment_seconds"],
            prefix=cfg["prefix"],
        )

    data_json.write_text(json.dumps({"data": data_entries}, indent=2))

    audio_conf = {
        "num_mel_bins": 128,
        "target_length": cfg["target_length"],
        "freqm": 0,
        "timem": 0,
        "mixup": 0,
        "dataset": "custom",
        "mode": "retrieval",
        "mean": DEFAULT_AUDIO_MEAN,
        "std": DEFAULT_AUDIO_STD,
        "noise": False,
        "im_res": 224,
        "frame_use": 5,
        "num_samples": None,
        "total_frame": DEFAULT_TOTAL_FRAMES,
    }

    dataset = dataloader_sync.AudiosetDataset(
        str(data_json),
        audio_conf=audio_conf,
        label_csv=str(label_csv),
    )
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=train_collate_fn)

    device = resolve_device()
    state = torch.load(DEFAULT_MODEL_PATH, map_location=device)
    num_register_tokens = infer_register_tokens(state)

    model = models.CAVMAESync(
        audio_length=audio_conf["target_length"],
        modality_specific_depth=11,
        num_register_tokens=num_register_tokens,
        cls_token=True,
        total_frame=audio_conf["total_frame"],
    )
    if not isinstance(model, torch.nn.DataParallel):
        model = torch.nn.DataParallel(model)
    for key in ("pos_embed_a", "pos_embed_v", "decoder_pos_embed_a", "decoder_pos_embed_v"):
        _resize_pos_embed(state, model.state_dict(), key)
        _resize_pos_embed(state, model.state_dict(), f"module.{key}")
    model.load_state_dict(state, strict=False)
    model = model.to(device)
    model.eval()

    # Collect per-frame embeddings for all segments
    all_audio = []
    all_video = []
    
    progress_interval = 50
    num_batches = len(loader)

    print(f"\n[{name}] Extracting  per-frame embeddings...")
    print(f"[{name}] Using forward_feat to get CLS tokens (global tokens per paper)")
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader, start=1):
            a_input, v_input, labels, video_ids, frame_indices = batch
            if batch_idx == 1 or batch_idx % progress_interval == 0 or batch_idx == num_batches:
                print(f"[{name}] Progress: {batch_idx}/{num_batches} batches")
            
            # Reshape inputs
            a_input = a_input.reshape(a_input.shape[0] * a_input.shape[1], a_input.shape[2], a_input.shape[3])
            v_input = v_input.reshape(a_input.shape[0], v_input.shape[2], v_input.shape[3], v_input.shape[4])

            audio_input, video_input = a_input.to(device), v_input.to(device)
            
            # CANONICAL extraction: Use forward_feat to get CLS tokens (global tokens)
            # This is how the paper intended the model to be used
            _, _, cls_a, cls_v = model.module.forward_feat(audio_input, video_input)
            
            # Normalize embeddings
            audio_emb = torch.nn.functional.normalize(cls_a, dim=-1)
            video_emb = torch.nn.functional.normalize(cls_v, dim=-1)

            # Reshape to (num_segments, num_frames, embed_dim)
            # Each segment has 16 frames, each frame has its own audio/video CLS token
            num_frames = audio_conf["total_frame"]
            audio_emb = audio_emb.view(len(video_ids), num_frames, -1).cpu().numpy().astype(np.float32)
            video_emb = video_emb.view(len(video_ids), num_frames, -1).cpu().numpy().astype(np.float32)

            all_audio.append(audio_emb)
            all_video.append(video_emb)

    # Concatenate all batches: (num_segments, num_frames, embed_dim)
    audio_perframe = np.concatenate(all_audio, axis=0)
    video_perframe = np.concatenate(all_video, axis=0)

    print(f"\n[{name}] ===== SAVING EMBEDDINGS =====")
    print(f"[{name}] Per-frame shape: {audio_perframe.shape} = (segments, frames, dim)")
    
    # Save per-frame embeddings (CANONICAL format per paper Section 3.5.1)
    np.save(output_root / "embeddings_audio_perframe.npy", audio_perframe)
    np.save(output_root / "embeddings_video_perframe.npy", video_perframe)
    
    print(f"[{name}] Saved CANONICAL per-frame embeddings:")
    print(f"[{name}]   - embeddings_audio_perframe.npy")
    print(f"[{name}]   - embeddings_video_perframe.npy")

    # Also save averaged versions for backward compatibility
    # WARNING: Averaging destroys the temporal alignment that the model learned!
    audio_avg = audio_perframe.mean(axis=1)
    video_avg = video_perframe.mean(axis=1)

    np.save(output_root / "embeddings_audio_averaged.npy", audio_avg)
    np.save(output_root / "embeddings_video_averaged.npy", video_avg)
    
    print(f"\n[{name}]  Saved AVERAGED embeddings (backward compat, NOT canonical):")
    print(f"[{name}]   - embeddings_audio_averaged.npy {audio_avg.shape}")
    print(f"[{name}]   - embeddings_video_averaged.npy {video_avg.shape}")
    print(f"[{name}]   WARNING: Averaging destroys temporal alignment!")
    print(f"[{name}]     Use *_perframe.npy for RSA and analysis\n")

    return output_root


for config_name in RUN_CONFIGS:
    run_config(config_name)


Run 1: 106 paired chunks
Run 2: 110 paired chunks
Run 3: 93 paired chunks
Run 4: 31 paired chunks
Run 5: 41 paired chunks
Run 6: 113 paired chunks
Run 7: 129 paired chunks
Run 8: 124 paired chunks
Run 9: 41 paired chunks
Run 10: 90 paired chunks
Run 11: 92 paired chunks
Run 12: 101 paired chunks
Run 13: 70 paired chunks
Run 14: 41 paired chunks
Run 15: 116 paired chunks
Run 16: 115 paired chunks
Run 17: 127 paired chunks
Run 18: 41 paired chunks
Total paired 2s segments: 1581
[ffmpeg] extracting frames: ffmpeg -i /home/amin/Research/Representation/Movie/data/Setareh/Data/Video17/Video17_chunks/Video17_part_091.mp4 -vf fps=8.0 -vframes 16 -q:v 2 /home/amin/Research/Representation/Movie/outputs/model_embeddings/cav-mae-sync/2s/frames_sync/segment_2s_1503/_tmp_frames/frame_%03d.jpg
[ffmpeg] extracting frames: ffmpeg -i /home/amin/Research/Representation/Movie/data/Setareh/Data/Video17/Video17_chunks/Video17_part_092.mp4 -vf fps=8.0 -vframes 16 -q:v 2 /home/amin/Research/Representation/Mov

/home/amin/miniconda3/envs/cav-mae-sync/lib/python3.7/site-packages/torchvision/transforms/transforms.py:330: UserWarning: Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. Please use InterpolationMode enum.
  "Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. "


A CAV-MAE Model
Use norm_pix_loss:  False
Learnable Positional Embedding:  False
Keep Register Tokens:  False
Number of Audio Patches: 96, Visual Patches: 196
Number of Registers: 8
Using CLS Token
Using 16 frames
Audio Positional Embedding Shape: torch.Size([1, 96, 768])
Visual Positional Embedding Shape: torch.Size([1, 196, 768])


/home/amin/miniconda3/envs/cav-mae-sync/lib/python3.7/site-packages/torch/cuda/__init__.py:155: UserWarning: 
NVIDIA GeForce RTX 5070 Ti Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_70 sm_75 sm_80 sm_86.
If you want to use the NVIDIA GeForce RTX 5070 Ti Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(incompatible_device_warn.format(device_name, capability, " ".join(arch_list), device_name))


[info] resized module.pos_embed_a from (1, 208, 768) to (1, 96, 768)
[info] resized module.decoder_pos_embed_a from (1, 208, 512) to (1, 96, 512)

[2s] Extracting  per-frame embeddings...
[2s] Using forward_feat to get CLS tokens (global tokens per paper)
[2s] Progress: 1/1581 batches
[2s] Progress: 50/1581 batches
[2s] Progress: 100/1581 batches
[2s] Progress: 150/1581 batches
[2s] Progress: 200/1581 batches
[2s] Progress: 250/1581 batches
[2s] Progress: 300/1581 batches
[2s] Progress: 350/1581 batches
[2s] Progress: 400/1581 batches
[2s] Progress: 450/1581 batches
[2s] Progress: 500/1581 batches
[2s] Progress: 550/1581 batches
[2s] Progress: 600/1581 batches
[2s] Progress: 650/1581 batches
[2s] Progress: 700/1581 batches
[2s] Progress: 750/1581 batches
[2s] Progress: 800/1581 batches
[2s] Progress: 850/1581 batches
[2s] Progress: 900/1581 batches
[2s] Progress: 950/1581 batches
[2s] Progress: 1000/1581 batches
[2s] Progress: 1050/1581 batches
[2s] Progress: 1100/1581 batches
[2s] Pro

In [ ]:
# ============================================================================
# Verify saved embeddings
# ============================================================================
CHECK_CONFIG = "2s"
OUTPUT_ROOT = DEFAULT_OUTPUTS_ROOT / CHECK_CONFIG

print("="*80)
print("VERIFYING SAVED EMBEDDINGS")
print("="*80)

# Load per-frame embeddings (CANONICAL)
audio_perframe = np.load(OUTPUT_ROOT / "embeddings_audio_perframe.npy")
video_perframe = np.load(OUTPUT_ROOT / "embeddings_video_perframe.npy")

print(f"\nCANONICAL per-frame embeddings:")
print(f"  Audio: {audio_perframe.shape} = (segments, frames, dim)")
print(f"  Video: {video_perframe.shape} = (segments, frames, dim)")

# Load averaged embeddings (backward compatibility)
audio_avg = np.load(OUTPUT_ROOT / "embeddings_audio_averaged.npy")
video_avg = np.load(OUTPUT_ROOT / "embeddings_video_averaged.npy")

print(f"\nAveraged embeddings (NOT canonical):")
print(f"  Audio: {audio_avg.shape} = (segments, dim)")
print(f"  Video: {video_avg.shape} = (segments, dim)")

# Verify averaging is correct
audio_avg_check = audio_perframe.mean(axis=1)
video_avg_check = video_perframe.mean(axis=1)
assert np.allclose(audio_avg, audio_avg_check), "Audio averaging mismatch!"
assert np.allclose(video_avg, video_avg_check), "Video averaging mismatch!"
print(f"\nVerified: Averaged embeddings match mean of per-frame")


VERIFYING SAVED EMBEDDINGS

✓ CANONICAL per-frame embeddings:
  Audio: (1581, 16, 768) = (segments, frames, dim)
  Video: (1581, 16, 768) = (segments, frames, dim)

✓ Averaged embeddings (NOT canonical):
  Audio: (1581, 768) = (segments, dim)
  Video: (1581, 768) = (segments, dim)

✓ Verified: Averaged embeddings match mean of per-frame

READY FOR ANALYSIS
Use embeddings_audio_perframe.npy and embeddings_video_perframe.npy for:
  1. Cross-modal similarity (diagonal method per paper)
  2. RSA with brain data (concatenate frames per segment)
  3. Any analysis requiring temporal information
